# GeoID Lookup — preset-driven anonymous access demo

Demonstrates the **lookup-only** access model enabled by the `geoid` preset:

- Setup applies the `geoid` catalog-tier preset (sysadmin token).
- Two items are ingested via STAC transactions (authenticated).
- The anonymous section shows:
  - `POST /search/catalogs/{cat}/geoid-search` (body `{"geoid": ...}`) — ALLOW
  - `POST /search/catalogs/{cat}/geoid-search` (body `{"external_id": ..., "collection_id": ...}`) — ALLOW
  - `GET /stac/catalogs/{cat}/collections/{coll}/items/{id}` — ALLOW (exact lookup)
  - Collection list / items list / STAC search — DENY (401/403)

Required env vars:
- `DYNASTORE_BASE_URL` — base URL (default `http://localhost:8080`)
- `DYNASTORE_SYSADMIN_TOKEN` — sysadmin bearer token for catalog creation and preset apply
- `DYNASTORE_TOKEN` — authenticated user token for item ingestion

In [ ]:
import json
import os
import time
import uuid

import httpx

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True), override=False)
except Exception:
    pass

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
SYSADMIN_TOKEN = os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
TOKEN = os.environ.get("DYNASTORE_TOKEN") or SYSADMIN_TOKEN
if not SYSADMIN_TOKEN:
    raise RuntimeError("Set DYNASTORE_SYSADMIN_TOKEN before running.")
if not TOKEN:
    raise RuntimeError("Set DYNASTORE_TOKEN before running.")

RUN_ID = uuid.uuid4().hex[:6]
CAT_ID = f"geoid-demo-{RUN_ID}"
COLL_ID = f"items-{RUN_ID}"

JSON_HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
SYSADMIN_HEADERS = {**JSON_HEADERS, "Authorization": f"Bearer {SYSADMIN_TOKEN}"}
AUTH_HEADERS = {**JSON_HEADERS, "Authorization": f"Bearer {TOKEN}"}

# sysadmin client — catalog creation + preset apply
admin = httpx.Client(base_url=BASE_URL, headers=SYSADMIN_HEADERS, timeout=30.0)
# authenticated client — item ingestion
client = httpx.Client(base_url=BASE_URL, headers=AUTH_HEADERS, timeout=30.0)
# anonymous client — lookup demo
anon = httpx.Client(base_url=BASE_URL, headers=JSON_HEADERS, timeout=30.0)

print(f"RUN_ID={RUN_ID}  catalog={CAT_ID}  collection={COLL_ID}")
print(f"base: {BASE_URL}")

## Setup — create catalog, apply `geoid` preset, ingest items

The `geoid` preset is a `CATALOG`-tier preset. It applies the `private_catalog` bundle (PG-first storage + per-tenant private ES, IAM DENY on enumeration routes) and opts the catalog in to anonymous lookup via `CatalogLookupAudience.is_public=True`.

### 1. Create catalog and wait for it to be ready

In [ ]:
r = admin.post("/stac/catalogs", content=json.dumps({
    "id": CAT_ID, "type": "Catalog", "stac_version": "1.0.0",
    "title": f"GeoID demo {RUN_ID}",
    "description": "Lookup-only demo — safe to delete",
    "links": [],
}))
assert r.status_code in (200, 201, 409), f"catalog create: {r.status_code} {r.text[:200]}"
print(f"POST /stac/catalogs: {r.status_code}")

# Poll until tenant is provisioned
deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    r = admin.get(f"/stac/catalogs/{CAT_ID}")
    if r.status_code in (200, 401, 403):
        break
    time.sleep(2)
else:
    raise TimeoutError(f"catalog {CAT_ID!r} not reachable after 60s")
print(f"Catalog {CAT_ID!r} ready (HTTP {r.status_code})")

### 2. Apply `geoid` preset

`POST /configs/catalogs/{cat}/presets/geoid` — catalog-tier preset, sysadmin token required.

In [ ]:
r = admin.post(f"/configs/catalogs/{CAT_ID}/presets/geoid")
assert r.is_success, f"apply geoid preset failed: {r.status_code} {r.text[:300]}"
print(f"POST /configs/catalogs/{CAT_ID}/presets/geoid: {r.status_code}")
print(json.dumps(r.json(), indent=2)[:600])

### 3. Create collection and ingest 2 items

In [ ]:
# Create collection (retry 409 while catalog finishes provisioning)
deadline = time.monotonic() + 120
delay = 1.0
while time.monotonic() < deadline:
    r = client.post(f"/stac/catalogs/{CAT_ID}/collections", content=json.dumps({
        "id": COLL_ID, "type": "Collection", "stac_version": "1.0.0",
        "description": "GeoID lookup demo items", "license": "proprietary",
        "extent": {
            "spatial": {"bbox": [[-180, -90, 180, 90]]},
            "temporal": {"interval": [[None, None]]},
        },
        "links": [],
    }))
    if r.is_success:
        print(f"collection {COLL_ID!r}: {r.status_code}")
        break
    if r.status_code == 409:
        time.sleep(delay)
        delay = min(delay * 1.7, 8.0)
        continue
    raise RuntimeError(f"collection POST failed: {r.status_code} {r.text[:300]}")
else:
    raise TimeoutError(f"collection {COLL_ID!r}: still 409 after 120s")

ITEM_IDS = []
for i in range(2):
    item_id = f"geoid-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{CAT_ID}/collections/{COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0", "id": item_id,
            "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.5, 41.9]},
            "bbox": [12.4 + i * 0.5, 41.8, 12.6 + i * 0.5, 42.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z"},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code}")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

assert len(ITEM_IDS) == 2, f"only {len(ITEM_IDS)}/2 items ingested"
GEOID_TO_LOOK_UP = ITEM_IDS[0]  # use ingested id as the geoid in lookup steps
print(f"\nWill look up geoid={GEOID_TO_LOOK_UP!r}")

---
## Anonymous lookups — allowed surfaces

The `geoid` preset opens `POST /search/catalogs/{cat}/geoid-search` and exact-item GET to anonymous callers while keeping all enumeration surfaces gated.

### 4. Lookup by geoid

`POST /search/catalogs/{cat}/geoid-search` with body `{"geoid": ...}`. Response model `GeoidCollection` exposes a `results` list.

In [ ]:
r = anon.post(
    f"/search/catalogs/{CAT_ID}/geoid-search",
    content=json.dumps({"geoid": GEOID_TO_LOOK_UP}),
)
print(f"POST /search/.../geoid-search (by geoid): HTTP {r.status_code}")
assert r.is_success, f"geoid lookup failed: {r.status_code} {r.text[:300]}"
body = r.json()
results = body.get("results", [])
assert len(results) >= 1, f"expected at least 1 result, got {len(results)}"
row = results[0]
print(f"  geoid      : {row.get('geoid')}")
print(f"  collection : {row.get('collection_id')}")
geom = row.get("geometry") or {}
print(f"  geometry   : {geom.get('type')}")
print("  OK — anonymous geoid lookup succeeded")

### 5. Lookup by external_id

`POST /search/catalogs/{cat}/geoid-search` with body `{"external_id": ..., "collection_id": ...}`. `collection_id` is required — external_id is not globally unique.

In [ ]:
r = anon.post(
    f"/search/catalogs/{CAT_ID}/geoid-search",
    content=json.dumps({"external_id": ITEM_IDS[1], "collection_id": COLL_ID}),
)
print(f"POST /search/.../geoid-search (by external_id): HTTP {r.status_code}")
assert r.is_success, f"external_id lookup failed: {r.status_code} {r.text[:300]}"
results = r.json().get("results", [])
assert len(results) >= 1, f"expected at least 1 result, got {len(results)}"
print(f"  resolved geoid: {results[0].get('geoid')}")
print("  OK — anonymous external_id lookup succeeded")

### 6. Exact-item GET via STAC

Anonymous callers may retrieve a known item by id via the canonical STAC item route.

In [ ]:
r = anon.get(f"/stac/catalogs/{CAT_ID}/collections/{COLL_ID}/items/{GEOID_TO_LOOK_UP}")
print(f"GET /stac/.../items/{GEOID_TO_LOOK_UP}: HTTP {r.status_code}")
assert r.is_success, f"exact-item GET failed: {r.status_code} {r.text[:300]}"
item = r.json()
print(f"  id  : {item.get('id')}")
print(f"  type: {item.get('type')}")
print("  OK — anonymous exact-item GET succeeded")

---
## Enumeration surfaces — denied

The `geoid` preset's IAM bundle blocks anonymous collection listing, items listing, and STAC search.

In [ ]:
def probe_deny(label, method, path, **kw):
    r = anon.request(method, path, **kw)
    blocked = r.status_code in (401, 403)
    flag = "DENY " if blocked else ("ALLOW" if r.is_success else "OTHER")
    print(f"  {flag}  {label:<30}  {method:<5}  {path[:70]}  -> HTTP {r.status_code}")
    return r, blocked

checks = [
    ("collection list", "GET",  f"/stac/catalogs/{CAT_ID}/collections",          {}),
    ("items list",      "GET",  f"/stac/catalogs/{CAT_ID}/collections/{COLL_ID}/items", {}),
    ("STAC search",     "POST", f"/stac/catalogs/{CAT_ID}/search",
     {"content": json.dumps({"limit": 10})}),
]
all_blocked = True
for label, method, path, kw in checks:
    _, blocked = probe_deny(label, method, path, **kw)
    if not blocked:
        all_blocked = False

assert all_blocked, "at least one enumeration surface was not blocked — review geoid preset IAM policies"
print("\n  OK — all enumeration surfaces denied")

---
## Recap

| Action | Preset | Anonymous |
|---|---|---|
| `POST /search/.../geoid-search` (`geoid`) | `geoid` | ALLOW |
| `POST /search/.../geoid-search` (`external_id` + `collection_id`) | `geoid` | ALLOW |
| `GET /stac/.../items/{id}` (exact) | `geoid` | ALLOW |
| Collection list / items list / STAC search | `geoid` | DENY |

## Teardown

In [ ]:
r = admin.delete(f"/stac/catalogs/{CAT_ID}", params={"force": "true"})
print(f"DELETE {CAT_ID!r}: {r.status_code}")
admin.close()
client.close()
anon.close()
print("Done.")